# Previsão Horária da Potência Ativa Residencial

## Etapa 1 — Preparação e Auditoria dos Dados

Objetivo: preparar a série temporal da variável Global_active_power
para previsão da potência ativa média da próxima hora.

Critério de validade horária:

- frequência original: 1 minuto;
- 60 medições esperadas por hora;
- mínimo de 54 medições válidas por hora (90%);
- horas abaixo desse limite serão consideradas inválidas.

In [0]:
# ------------------------------------------------------------
# Carga do dataset bruto (UCI Household Power Consumption)
# Arquivo CSV separado por ';' com '?' como marcador de nulo
# ------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

file_path = "/Volumes/workspace/default/tcc/household_power_consumption.txt"

df_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("nullValue", "?")
    .option("inferSchema", False)
    .csv(file_path)
)

print(f"Total de registros: {df_raw.count():,}")
df_raw.printSchema()
display(df_raw.limit(5))

In [0]:
# ------------------------------------------------------------
# Conversão de tipos: cria timestamp único e cast de colunas numéricas
# ------------------------------------------------------------
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df = df_raw.withColumn(
    "datetime",
    F.to_timestamp(
        F.concat_ws(" ", F.col("Date"), F.col("Time")),
        "d/M/yyyy H:mm:ss"
    )
)

for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))

display(
    df.select(
        "datetime",
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity"
    ).limit(10)
)

In [0]:
# ------------------------------------------------------------
# Auditoria inicial: total de registros, período coberto e
# contagem de timestamps/potência ausentes
# ------------------------------------------------------------
audit = df.agg(
    F.count("*").alias("total_registros"),
    F.min("datetime").alias("inicio"),
    F.max("datetime").alias("fim"),
    F.sum(
        F.when(F.col("datetime").isNull(), 1).otherwise(0)
    ).alias("timestamp_invalido"),
    F.sum(
        F.when(F.col("Global_active_power").isNull(), 1).otherwise(0)
    ).alias("global_active_power_nulo")
)

display(audit)

In [0]:
# Quantifica valores ausentes em Global_active_power
# (variável-alvo do estudo)
total = df.count()

gap_null = df.filter(
    F.col("Global_active_power").isNull()
).count()

print(f"Total de registros: {total:,}")
print(f"Global_active_power ausente: {gap_null:,}")
print(f"Percentual ausente: {gap_null / total * 100:.4f}%")

In [0]:
# Verifica se existem timestamps duplicados
# (mesmo instante registrado mais de uma vez)
duplicates = (
    df
    .filter(F.col("datetime").isNotNull())
    .groupBy("datetime")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_timestamps = duplicates.count()

print(f"Timestamps duplicados: {duplicate_timestamps:,}")

if duplicate_timestamps > 0:
    display(
        duplicates
        .orderBy(F.desc("count"))
        .limit(20)
    )

In [0]:
# ------------------------------------------------------------
# Agregação horária: agrupa medições por hora truncada e calcula
# média de Global_active_power e contagem de minutos válidos
# ------------------------------------------------------------
df_hour_base = (
    df
    .filter(F.col("datetime").isNotNull())
    .withColumn(
        "hour_ts",
        F.date_trunc("hour", F.col("datetime"))
    )
)

hourly_raw = (
    df_hour_base
    .groupBy("hour_ts")
    .agg(
        F.count("Global_active_power").alias("valid_minutes"),
        F.avg("Global_active_power").alias("global_active_power_mean")
    )
    .orderBy("hour_ts")
)

display(hourly_raw.limit(24))

In [0]:
# ------------------------------------------------------------
# Construção do calendário horário completo: gera todas as horas
# entre o primeiro e o último timestamp, faz left join com as
# horas agregadas e marca validade (>= 54 minutos válidos)
# ------------------------------------------------------------
bounds = df.agg(
    F.min("datetime").alias("min_dt"),
    F.max("datetime").alias("max_dt")
)

hour_calendar = bounds.select(
    F.explode(
        F.sequence(
            F.date_trunc("hour", F.col("min_dt")),
            F.date_trunc("hour", F.col("max_dt")),
            F.expr("INTERVAL 1 HOUR")
        )
    ).alias("hour_ts")
)

hourly_complete = (
    hour_calendar
    .join(hourly_raw, on="hour_ts", how="left")
    .withColumn(
        "valid_minutes",
        F.coalesce(F.col("valid_minutes"), F.lit(0))
    )
    .withColumn(
        "valid_hour",
        F.col("valid_minutes") >= 54
    )
    .orderBy("hour_ts")
)

display(hourly_complete.limit(24))

In [0]:
# Resumo da auditoria horária: horas esperadas, válidas,
# inválidas e taxa de aproveitamento
total_expected_hours = hourly_complete.count()

valid_hours = hourly_complete.filter(
    F.col("valid_hour") == True
).count()

invalid_hours = total_expected_hours - valid_hours

completely_missing = hourly_complete.filter(
    F.col("valid_minutes") == 0
).count()

print(f"Horas esperadas no período: {total_expected_hours:,}")
print(f"Horas válidas (>=54 minutos): {valid_hours:,}")
print(f"Horas inválidas (<54 minutos): {invalid_hours:,}")
print(f"Horas totalmente sem medições válidas: {completely_missing:,}")
print(f"Taxa de aproveitamento: {valid_hours / total_expected_hours * 100:.2f}%")

In [0]:
# Filtra apenas as horas válidas para uso no modelo
hourly_valid = (
    hourly_complete
    .filter(F.col("valid_hour") == True)
    .select(
        "hour_ts",
        "global_active_power_mean",
        "valid_minutes"
    )
    .orderBy("hour_ts")
)

display(hourly_valid.limit(24))

In [0]:
# Estatísticas descritivas da potência ativa média horária
display(
    hourly_valid.select(
        "global_active_power_mean"
    ).summary()
)

In [0]:
# ------------------------------------------------------------
# Perfil por hora do dia: média e contagem de observações
# ------------------------------------------------------------
by_hour = (
    hourly_valid
    .withColumn(
        "hour_of_day",
        F.hour("hour_ts")
    )
    .groupBy("hour_of_day")
    .agg(
        F.avg("global_active_power_mean").alias("mean_active_power"),
        F.count("*").alias("observations")
    )
    .orderBy("hour_of_day")
)

display(by_hour)

In [0]:
# ------------------------------------------------------------
# Perfil por dia da semana: média e contagem de observações
# ------------------------------------------------------------
by_weekday = (
    hourly_valid
    .withColumn(
        "weekday_number",
        F.dayofweek("hour_ts")
    )
    .withColumn(
        "weekday",
        F.date_format("hour_ts", "EEEE")
    )
    .groupBy(
        "weekday_number",
        "weekday"
    )
    .agg(
        F.avg("global_active_power_mean").alias("mean_active_power"),
        F.count("*").alias("observations")
    )
    .orderBy("weekday_number")
)

display(by_weekday)

In [0]:
# ------------------------------------------------------------
# Etapa 2 — Construção da base horária para modelagem
# Mantém power = None para horas inválidas (preserva o calendário)
# ------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math

hourly_model_base = (
    hourly_complete
    .withColumn(
        "power",
        F.when(
            F.col("valid_hour") == True,
            F.col("global_active_power_mean")
        ).otherwise(F.lit(None).cast("double"))
    )
    .select(
        "hour_ts",
        "power",
        "valid_minutes",
        "valid_hour"
    )
    .orderBy("hour_ts")
)

display(hourly_model_base.limit(30))

In [0]:
# Verifica continuidade do calendário horário:
# se a diferença entre horas consecutivas não for 1h, há falha
w_time = Window.orderBy("hour_ts")

calendar_check = (
    hourly_model_base
    .withColumn(
        "previous_hour",
        F.lag("hour_ts", 1).over(w_time)
    )
    .withColumn(
        "difference_hours",
        (
            F.unix_timestamp("hour_ts") -
            F.unix_timestamp("previous_hour")
        ) / 3600
    )
)

calendar_gaps = (
    calendar_check
    .filter(
        F.col("previous_hour").isNotNull() &
        (F.col("difference_hours") != 1)
    )
    .count()
)

print(f"Quebras no calendário horário: {calendar_gaps}")

In [0]:
# ------------------------------------------------------------
# Etapa 3 — Engenharia de features: lags temporais
# lag_1 (hora anterior), lag_24 (mesma hora do dia anterior),
# lag_168 (mesma hora da semana anterior)
# ------------------------------------------------------------
w = Window.orderBy("hour_ts")

df_features = (
    hourly_model_base
    .withColumn(
        "lag_1",
        F.lag("power", 1).over(w)
    )
    .withColumn(
        "lag_24",
        F.lag("power", 24).over(w)
    )
    .withColumn(
        "lag_168",
        F.lag("power", 168).over(w)
    )
)

display(
    df_features.select(
        "hour_ts",
        "power",
        "lag_1",
        "lag_24",
        "lag_168"
    ).limit(200)
)

In [0]:
# ------------------------------------------------------------
# Variáveis calendário: hora do dia, dia da semana (0=segunda)
# e mês — usadas para codificação cíclica e exploração
# ------------------------------------------------------------
df_features = (
    df_features
    .withColumn(
        "hour_of_day",
        F.hour("hour_ts")
    )
    .withColumn(
        "day_of_week",
        F.pmod(
            F.dayofweek("hour_ts") + F.lit(5),
            F.lit(7)
        )
    )
    .withColumn(
        "month",
        F.month("hour_ts")
    )
)

display(
    df_features.select(
        "hour_ts",
        "hour_of_day",
        "day_of_week",
        "month"
    ).limit(30)
)

In [0]:
# Codificação cíclica seno/cosseno para hora do dia e dia da semana
# (preserva a continuidade circular: 23h → 0h)
df_features = (
    df_features
    .withColumn(
        "hour_sin",
        F.sin(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "hour_cos",
        F.cos(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "dow_sin",
        F.sin(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
    .withColumn(
        "dow_cos",
        F.cos(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
)

In [0]:
# Codificação cíclica seno/cosseno para mês
# (preserva continuidade circular: dez → jan)
df_features = (
    df_features
    .withColumn(
        "month_sin",
        F.sin(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
    .withColumn(
        "month_cos",
        F.cos(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
)

In [0]:
# ------------------------------------------------------------
# Média móvel 24h: usa janela deslizante das 24 horas anteriores
# Só é calculada quando há exatamente 24 valores válidos
# ------------------------------------------------------------
w_roll_24 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-24, -1)
)

df_features = (
    df_features
    .withColumn(
        "rolling_count_24",
        F.count("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24_raw",
        F.avg("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24",
        F.when(
            F.col("rolling_count_24") == 24,
            F.col("rolling_mean_24_raw")
        )
    )
)

In [0]:
# ------------------------------------------------------------
# Média móvel 168h (1 semana): usa janela das 168 horas anteriores
# Só é calculada quando há exatamente 168 valores válidos
# ------------------------------------------------------------
w_roll_168 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-168, -1)
)

df_features = (
    df_features
    .withColumn(
        "rolling_count_168",
        F.count("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168_raw",
        F.avg("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168",
        F.when(
            F.col("rolling_count_168") == 168,
            F.col("rolling_mean_168_raw")
        )
    )
)

In [0]:
# Auditoria das features: quantas observações têm cada feature válida
feature_quality = df_features.agg(
    F.count("*").alias("total_hours"),

    F.count("power").alias("target_valid"),

    F.count("lag_1").alias("lag_1_valid"),
    F.count("lag_24").alias("lag_24_valid"),
    F.count("lag_168").alias("lag_168_valid"),

    F.count("rolling_mean_24").alias("rolling_24_valid"),
    F.count("rolling_mean_168").alias("rolling_168_valid")
)

display(feature_quality)

In [0]:
# ------------------------------------------------------------
# Dataset principal: apenas observações com target e três lags válidos
# ------------------------------------------------------------
model_core = (
    df_features
    .filter(
        F.col("power").isNotNull() &
        F.col("lag_1").isNotNull() &
        F.col("lag_24").isNotNull() &
        F.col("lag_168").isNotNull()
    )
)

core_count = model_core.count()

print(f"Observações disponíveis no dataset principal: {core_count:,}")

In [0]:
# Aproveitamento: % do dataset principal em relação a todas as
# horas com target válido
valid_target_count = (
    df_features
    .filter(F.col("power").isNotNull())
    .count()
)

print(
    f"Aproveitamento após exigir os três lags: "
    f"{core_count / valid_target_count * 100:.2f}%"
)

In [0]:
# Subconjunto com médias móveis completas (24h e 168h)
model_with_rolling = (
    model_core
    .filter(
        F.col("rolling_mean_24").isNotNull() &
        F.col("rolling_mean_168").isNotNull()
    )
)

rolling_count = model_with_rolling.count()

print(
    f"Observações com lags + médias móveis completas: "
    f"{rolling_count:,}"
)

print(
    f"Aproveitamento em relação ao dataset principal: "
    f"{rolling_count / core_count * 100:.2f}%"
)

In [0]:
# Pré-visualização do dataset final com todas as features
display(
    model_core.select(
        "hour_ts",
        "power",
        "lag_1",
        "lag_24",
        "lag_168",
        "hour_of_day",
        "day_of_week",
        "month",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
        "rolling_mean_24",
        "rolling_mean_168"
    ).limit(20)
)

In [0]:
# ------------------------------------------------------------
# Etapa 4 — Divisão temporal treino / validação / teste
# Ordenação cronológica; sem embaralhamento (série temporal)
# ------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

model_data = (
    model_core
    .orderBy("hour_ts")
)

print(f"Total de observações disponíveis: {model_data.count():,}")

In [0]:
# Índice cronológico e pontos de corte:
# 64% treino, 16% validação, 20% teste
w_order = Window.orderBy("hour_ts")

model_indexed = (
    model_data
    .withColumn(
        "row_number",
        F.row_number().over(w_order)
    )
)

total_model = model_indexed.count()

train_end = int(total_model * 0.64)
validation_end = int(total_model * 0.80)

print(f"Total: {total_model:,}")
print(f"Fim do treino: {train_end:,}")
print(f"Fim da validação: {validation_end:,}")

In [0]:
# Particionamento estrito por ordem temporal
train_df = (
    model_indexed
    .filter(F.col("row_number") <= train_end)
)

validation_df = (
    model_indexed
    .filter(
        (F.col("row_number") > train_end) &
        (F.col("row_number") <= validation_end)
    )
)

test_df = (
    model_indexed
    .filter(F.col("row_number") > validation_end)
)

print(f"Treino: {train_df.count():,}")
print(f"Validação: {validation_df.count():,}")
print(f"Teste: {test_df.count():,}")

In [0]:
# Exibe período e contagem de cada partição
def show_period(name, dataframe):
    period = dataframe.agg(
        F.min("hour_ts").alias("inicio"),
        F.max("hour_ts").alias("fim"),
        F.count("*").alias("observacoes")
    )

    print(name)
    display(period)

show_period("TREINO", train_df)
show_period("VALIDAÇÃO", validation_df)
show_period("TESTE", test_df)

In [0]:
# ------------------------------------------------------------
# Etapa 5 — Baselines sazonais (naive)
# pred_daily = valor da mesma hora do dia anterior (lag_24)
# pred_weekly = valor da mesma hora da semana anterior (lag_168)
# ------------------------------------------------------------
baseline_validation = (
    validation_df
    .select(
        "hour_ts",
        F.col("power").alias("real"),
        F.col("lag_24").alias("pred_daily"),
        F.col("lag_168").alias("pred_weekly")
    )
)

display(baseline_validation.limit(20))

In [0]:
# Métricas MAE e RMSE para baseline diário (lag_24) na validação
daily_metrics = baseline_validation.agg(
    F.avg(
        F.abs(
            F.col("real") - F.col("pred_daily")
        )
    ).alias("MAE"),

    F.sqrt(
        F.avg(
            F.pow(
                F.col("real") - F.col("pred_daily"),
                2
            )
        )
    ).alias("RMSE")
)

print("BASELINE DIÁRIO — VALIDAÇÃO")
display(daily_metrics)

In [0]:
# Métricas MAE e RMSE para baseline semanal (lag_168) na validação
weekly_metrics = baseline_validation.agg(
    F.avg(
        F.abs(
            F.col("real") - F.col("pred_weekly")
        )
    ).alias("MAE"),

    F.sqrt(
        F.avg(
            F.pow(
                F.col("real") - F.col("pred_weekly"),
                2
            )
        )
    ).alias("RMSE")
)

print("BASELINE SEMANAL — VALIDAÇÃO")
display(weekly_metrics)

In [0]:
# Tabela comparativa dos baselines sazonais
daily_result = (
    daily_metrics
    .withColumn("Modelo", F.lit("Naive sazonal 24h"))
)

weekly_result = (
    weekly_metrics
    .withColumn("Modelo", F.lit("Naive sazonal 168h"))
)

baseline_comparison = (
    daily_result
    .unionByName(weekly_result)
    .select(
        "Modelo",
        "MAE",
        "RMSE"
    )
    .orderBy("RMSE")
)

display(baseline_comparison)

In [0]:
# Verifica se statsmodels já está instalado
import statsmodels
print("statsmodels:", statsmodels.__version__)

In [0]:
# Instala statsmodels para modelagem SARIMA
%pip install statsmodels

In [0]:
# Reinicia o kernel Python para carregar statsmodels recém-instalado
%restart_python


In [0]:
# [OBSOLETO] Tentativa de extrair limites temporais após restart — falhou por perda de variáveis
# train_period = train_df.agg(

In [0]:
# Reinicia o Python via dbutils para recarregar módulos
# [OBSOLETO] substituído pela célula de restauração completa abaixo
# dbutils.library.restartPython()


In [0]:
# [OBSOLETO] Mesma tentativa de extração de limites — falhou pela mesma razão
# train_period = train_df.agg(

In [0]:
# [OBSOLETO] Tentativa intermediária de recriar divisão treino/validação
# — falhou por dependência de variáveis perdidas no restart
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window
#
# # Base já preparada na etapa anterior
# model_data = model_core.orderBy("hour_ts")
#
# # Criar índice cronológico
# w_order = Window.orderBy("hour_ts")
#
# model_indexed = (
#     model_data
#     .withColumn(
#         "row_number",
#         F.row_number().over(w_order)
#     )
# )
#
# total_model = model_indexed.count()
#
# train_end = int(total_model * 0.64)
# validation_end = int(total_model * 0.80)
#
# # Divisão cronológica
# train_df = (
#     model_indexed
#     .filter(F.col("row_number") <= train_end)
# )
#
# validation_df = (
#     model_indexed
#     .filter(
#         (F.col("row_number") > train_end) &
#         (F.col("row_number") <= validation_end)
#     )
# )
#
# test_df = (
#     model_indexed
#     .filter(F.col("row_number") > validation_end)
# )
#
# print(f"Treino: {train_df.count():,}")
# print(f"Validação: {validation_df.count():,}")
# print(f"Teste: {test_df.count():,}")

In [0]:
# ============================================================
# RESTAURAÇÃO DO AMBIENTE — TCC PREVISÃO HORÁRIA
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
import math

# ------------------------------------------------------------
# 1. CARREGAR O DATASET ORIGINAL
# ------------------------------------------------------------

file_path = "/Volumes/workspace/default/tcc/household_power_consumption.txt"

df_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("nullValue", "?")
    .option("inferSchema", False)
    .csv(file_path)
)

# ------------------------------------------------------------
# 2. CONVERTER DATA/HORA E VARIÁVEIS NUMÉRICAS
# ------------------------------------------------------------

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df = df_raw.withColumn(
    "datetime",
    F.to_timestamp(
        F.concat_ws(" ", F.col("Date"), F.col("Time")),
        "d/M/yyyy H:mm:ss"
    )
)

for c in numeric_cols:
    df = df.withColumn(
        c,
        F.col(c).cast(DoubleType())
    )

# ------------------------------------------------------------
# 3. AGREGAR PARA ESCALA HORÁRIA
# ------------------------------------------------------------

df_hour_base = (
    df
    .filter(F.col("datetime").isNotNull())
    .withColumn(
        "hour_ts",
        F.date_trunc("hour", F.col("datetime"))
    )
)

hourly_raw = (
    df_hour_base
    .groupBy("hour_ts")
    .agg(
        F.count("Global_active_power").alias("valid_minutes"),
        F.avg("Global_active_power").alias(
            "global_active_power_mean"
        )
    )
)

# ------------------------------------------------------------
# 4. CRIAR CALENDÁRIO HORÁRIO COMPLETO
# ------------------------------------------------------------

bounds = df.agg(
    F.min("datetime").alias("min_dt"),
    F.max("datetime").alias("max_dt")
)

hour_calendar = bounds.select(
    F.explode(
        F.sequence(
            F.date_trunc("hour", F.col("min_dt")),
            F.date_trunc("hour", F.col("max_dt")),
            F.expr("INTERVAL 1 HOUR")
        )
    ).alias("hour_ts")
)

hourly_complete = (
    hour_calendar
    .join(
        hourly_raw,
        on="hour_ts",
        how="left"
    )
    .withColumn(
        "valid_minutes",
        F.coalesce(
            F.col("valid_minutes"),
            F.lit(0)
        )
    )
    .withColumn(
        "valid_hour",
        F.col("valid_minutes") >= 54
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 5. CRIAR A SÉRIE HORÁRIA OFICIAL
# Horas inválidas permanecem no calendário, mas power = NULL
# ------------------------------------------------------------

hourly_model_base = (
    hourly_complete
    .withColumn(
        "power",
        F.when(
            F.col("valid_hour") == True,
            F.col("global_active_power_mean")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
    .select(
        "hour_ts",
        "power",
        "valid_minutes",
        "valid_hour"
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 6. CRIAR LAGS
# ------------------------------------------------------------

w = Window.orderBy("hour_ts")

df_features = (
    hourly_model_base
    .withColumn(
        "lag_1",
        F.lag("power", 1).over(w)
    )
    .withColumn(
        "lag_24",
        F.lag("power", 24).over(w)
    )
    .withColumn(
        "lag_168",
        F.lag("power", 168).over(w)
    )
)

# ------------------------------------------------------------
# 7. ATRIBUTOS TEMPORAIS
# ------------------------------------------------------------

df_features = (
    df_features
    .withColumn(
        "hour_of_day",
        F.hour("hour_ts")
    )
    .withColumn(
        "day_of_week",
        F.pmod(
            F.dayofweek("hour_ts") + F.lit(5),
            F.lit(7)
        )
    )
    .withColumn(
        "month",
        F.month("hour_ts")
    )
)

# ------------------------------------------------------------
# 8. ATRIBUTOS CÍCLICOS
# ------------------------------------------------------------

df_features = (
    df_features
    .withColumn(
        "hour_sin",
        F.sin(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "hour_cos",
        F.cos(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "dow_sin",
        F.sin(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
    .withColumn(
        "dow_cos",
        F.cos(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
    .withColumn(
        "month_sin",
        F.sin(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
    .withColumn(
        "month_cos",
        F.cos(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
)

# ------------------------------------------------------------
# 9. MÉDIAS MÓVEIS SEM DATA LEAKAGE
# A janela termina sempre em t-1
# ------------------------------------------------------------

w_roll_24 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-24, -1)
)

w_roll_168 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-168, -1)
)

df_features = (
    df_features
    .withColumn(
        "rolling_count_24",
        F.count("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24_raw",
        F.avg("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24",
        F.when(
            F.col("rolling_count_24") == 24,
            F.col("rolling_mean_24_raw")
        )
    )
    .withColumn(
        "rolling_count_168",
        F.count("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168_raw",
        F.avg("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168",
        F.when(
            F.col("rolling_count_168") == 168,
            F.col("rolling_mean_168_raw")
        )
    )
)

# ------------------------------------------------------------
# 10. DATASET PRINCIPAL
# ------------------------------------------------------------

model_core = (
    df_features
    .filter(
        F.col("power").isNotNull() &
        F.col("lag_1").isNotNull() &
        F.col("lag_24").isNotNull() &
        F.col("lag_168").isNotNull()
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 11. DIVISÃO CRONOLÓGICA
# 64% treino / 16% validação / 20% teste
# ------------------------------------------------------------

w_order = Window.orderBy("hour_ts")

model_indexed = (
    model_core
    .withColumn(
        "row_number",
        F.row_number().over(w_order)
    )
)

total_model = model_indexed.count()

train_end = int(total_model * 0.64)
validation_end = int(total_model * 0.80)

train_df = (
    model_indexed
    .filter(
        F.col("row_number") <= train_end
    )
)

validation_df = (
    model_indexed
    .filter(
        (F.col("row_number") > train_end) &
        (F.col("row_number") <= validation_end)
    )
)

test_df = (
    model_indexed
    .filter(
        F.col("row_number") > validation_end
    )
)

# ------------------------------------------------------------
# 12. MATERIALIZAR E CONFERIR
# ------------------------------------------------------------

train_count = train_df.count()
validation_count = validation_df.count()
test_count = test_df.count()

print("==========================================")
print("AMBIENTE RESTAURADO COM SUCESSO")
print("==========================================")
print(f"Dataset original: {df_raw.count():,}")
print(f"Horas calendário: {hourly_complete.count():,}")
print(f"Model core: {total_model:,}")
print()
print(f"Treino:    {train_count:,}")
print(f"Validação: {validation_count:,}")
print(f"Teste:     {test_count:,}")

In [0]:
# Extração dos limites temporais de treino e validação
# (executada após a restauração completa do ambiente)
train_period = train_df.agg(
    F.min("hour_ts").alias("train_start"),
    F.max("hour_ts").alias("train_end_ts")
).collect()[0]

train_start = train_period["train_start"]
train_end_ts = train_period["train_end_ts"]

validation_period = validation_df.agg(
    F.min("hour_ts").alias("validation_start"),
    F.max("hour_ts").alias("validation_end_ts")
).collect()[0]

validation_start = validation_period["validation_start"]
validation_end_ts = validation_period["validation_end_ts"]

print(f"Treino: {train_start} a {train_end_ts}")
print(f"Validação: {validation_start} a {validation_end_ts}")

In [0]:
# ------------------------------------------------------------
# Filtra séries temporais para SARIMA: inclui calendário completo
# (com NaNs) para preservar frequência horária
# ------------------------------------------------------------
sarima_train_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(train_start)) &
        (F.col("hour_ts") <= F.lit(train_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)

sarima_validation_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(validation_start)) &
        (F.col("hour_ts") <= F.lit(validation_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)

print("Horas calendário no treino:", sarima_train_spark.count())
print("Horas calendário na validação:", sarima_validation_spark.count())

In [0]:
# Converte para pandas e define frequência horária ('h')
# NaNs são preenchidos por interpolação para o SARIMA
import pandas as pd
import numpy as np

train_pd = sarima_train_spark.toPandas()
validation_calendar_pd = sarima_validation_spark.toPandas()

train_pd["hour_ts"] = pd.to_datetime(train_pd["hour_ts"])
validation_calendar_pd["hour_ts"] = pd.to_datetime(
    validation_calendar_pd["hour_ts"]
)

train_series = (
    train_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

validation_series = (
    validation_calendar_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

print("Treino:", len(train_series))
print("Valores ausentes treino:", train_series.isna().sum())

print("\nValidação calendário:", len(validation_series))
print(
    "Valores ausentes validação:",
    validation_series.isna().sum()
)

In [0]:
# DataFrame de avaliação: apenas horas com target válido na validação
validation_eval_pd = (
    validation_df
    .select("hour_ts", "power")
    .orderBy("hour_ts")
    .toPandas()
)

validation_eval_pd["hour_ts"] = pd.to_datetime(
    validation_eval_pd["hour_ts"]
)

print(
    "Observações que serão usadas na avaliação:",
    len(validation_eval_pd)
)

In [0]:
# Função auxiliar: calcula MAE e RMSE
def evaluate_forecast(real, predicted):
    error = real - predicted

    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))

    return mae, rmse

In [0]:
# ------------------------------------------------------------
# SARIMA 1: ordem (1,0,1), sazonalidade (1,0,0,24) — ciclo diário
# Treinamento com maxiter=50 e sem impor estacionariedade/invertibilidade
# ------------------------------------------------------------
from statsmodels.tsa.statespace.sarimax import SARIMAX
import time

start_time = time.time()

sarima_1 = SARIMAX(
    train_series,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 0, 24),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_1_result = sarima_1.fit(
    disp=False,
    maxiter=50
)

elapsed = time.time() - start_time

print(f"Tempo de treinamento: {elapsed:.1f} segundos")
print(f"AIC: {sarima_1_result.aic:.2f}")
print(f"BIC: {sarima_1_result.bic:.2f}")

In [0]:
# Previsão multi-step para todo o período de validação
forecast_1 = sarima_1_result.get_forecast(
    steps=len(validation_series)
).predicted_mean

forecast_1.index = validation_series.index

In [0]:
# Converte previsão para DataFrame pandas
forecast_1_pd = pd.DataFrame({
    "hour_ts": forecast_1.index,
    "sarima_prediction": forecast_1.values
})

In [0]:
# Avaliação do SARIMA 1 na validação (apenas horas com target)
eval_1 = validation_eval_pd.merge(
    forecast_1_pd,
    on="hour_ts",
    how="inner"
)

mae_1, rmse_1 = evaluate_forecast(
    eval_1["power"].values,
    eval_1["sarima_prediction"].values
)

print("SARIMA 1 — (1,0,1)(1,0,0,24)")
print(f"MAE:  {mae_1:.4f}")
print(f"RMSE: {rmse_1:.4f}")
print(f"N avaliação: {len(eval_1):,}")

In [0]:
# Verificação de convergência da otimização MLE
print(sarima_1_result.mle_retvals)

In [0]:
# ------------------------------------------------------------
# Experimento SARIMA: testa 3 configurações de hiperparâmetros
# SARIMA_1: (1,0,1)(1,0,0,24) — base
# SARIMA_2: (2,0,1)(1,0,0,24) — AR adicional
# SARIMA_3: (1,0,1)(1,1,0,24) — diferenciação sazonal
# ------------------------------------------------------------
candidates = [
    {
        "name": "SARIMA_1",
        "order": (1, 0, 1),
        "seasonal_order": (1, 0, 0, 24)
    },
    {
        "name": "SARIMA_2",
        "order": (2, 0, 1),
        "seasonal_order": (1, 0, 0, 24)
    },
    {
        "name": "SARIMA_3",
        "order": (1, 0, 1),
        "seasonal_order": (1, 1, 0, 24)
    }
]

sarima_results = []

for candidate in candidates:

    print("\n-----------------------------------")
    print(candidate["name"])
    print("-----------------------------------")

    start_time = time.time()

    model = SARIMAX(
        train_series,
        order=candidate["order"],
        seasonal_order=candidate["seasonal_order"],
        trend="c" if candidate["seasonal_order"][1] == 0 else None,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    result = model.fit(
        disp=False,
        maxiter=50
    )

    forecast = result.get_forecast(
        steps=len(validation_series)
    ).predicted_mean

    forecast.index = validation_series.index

    forecast_pd = pd.DataFrame({
        "hour_ts": forecast.index,
        "prediction": forecast.values
    })

    evaluation = validation_eval_pd.merge(
        forecast_pd,
        on="hour_ts",
        how="inner"
    )

    mae, rmse = evaluate_forecast(
        evaluation["power"].values,
        evaluation["prediction"].values
    )

    elapsed = time.time() - start_time

    sarima_results.append({
        "Modelo": candidate["name"],
        "Order": str(candidate["order"]),
        "Seasonal": str(candidate["seasonal_order"]),
        "MAE": mae,
        "RMSE": rmse,
        "AIC": result.aic,
        "BIC": result.bic,
        "Convergiu": result.mle_retvals.get(
            "converged",
            None
        ),
        "Tempo_segundos": elapsed
    })

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(
        "Convergiu:",
        result.mle_retvals.get("converged")
    )

In [0]:
# Tabela comparativa dos modelos SARIMA ordenada por RMSE
sarima_comparison = (
    pd.DataFrame(sarima_results)
    .sort_values("RMSE")
)

display(sarima_comparison)

In [0]:
# Comparação final: baselines vs SARIMA na validação
baseline_168_mae = 0.5275
baseline_168_rmse = 0.7970

comparison_rows = [{
    "Modelo": "Baseline sazonal 168h",
    "MAE": baseline_168_mae,
    "RMSE": baseline_168_rmse
}]

for row in sarima_results:
    comparison_rows.append({
        "Modelo": row["Modelo"],
        "MAE": row["MAE"],
        "RMSE": row["RMSE"]
    })

full_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("RMSE")
)

display(full_comparison)

In [0]:
# ------------------------------------------------------------
# Etapa 5b — Modelo HistGradientBoosting (scikit-learn)
# Importa bibliotecas para ML tabular e métricas
# ------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

print("Bibliotecas carregadas.")

In [0]:
# Definição das features (lags, médias móveis e codificação cíclica)
# e variável-alvo (potência ativa média horária)
feature_cols = [
    "lag_1",
    "lag_24",
    "lag_168",

    "rolling_mean_24",
    "rolling_mean_168",

    "hour_sin",
    "hour_cos",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos"
]

target_col = "power"

In [0]:
# ------------------------------------------------------------
# Conversão para pandas: separa treino e validação com features
# ------------------------------------------------------------
train_ml_pd = (
    train_df
    .select(
        "hour_ts",
        target_col,
        *feature_cols
    )
    .orderBy("hour_ts")
    .toPandas()
)

validation_ml_pd = (
    validation_df
    .select(
        "hour_ts",
        target_col,
        *feature_cols
    )
    .orderBy("hour_ts")
    .toPandas()
)

print("Treino:", len(train_ml_pd))
print("Validação:", len(validation_ml_pd))

In [0]:
# Verifica valores ausentes nas features (HGB lida com NaN nativamente)
print("Valores ausentes no TREINO:")
print(train_ml_pd[feature_cols].isna().sum())

print("\nValores ausentes na VALIDAÇÃO:")
print(validation_ml_pd[feature_cols].isna().sum())

In [0]:
# Separa X e y para treino e validação
X_train = train_ml_pd[feature_cols]
y_train = train_ml_pd[target_col]

X_val = validation_ml_pd[feature_cols]
y_val = validation_ml_pd[target_col]

print(X_train.shape)
print(X_val.shape)

In [0]:
# ------------------------------------------------------------
# HGB 1: configuração inicial — lr=0.05, 200 iterações
# ------------------------------------------------------------
hgb_1 = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.0,
    early_stopping=False,
    random_state=42
)

start = time.time()

hgb_1.fit(
    X_train,
    y_train
)

elapsed = time.time() - start

pred_1 = hgb_1.predict(X_val)

mae_1 = mean_absolute_error(
    y_val,
    pred_1
)

rmse_1 = np.sqrt(
    mean_squared_error(
        y_val,
        pred_1
    )
)

print("HistGradientBoosting 1")
print(f"MAE:  {mae_1:.4f}")
print(f"RMSE: {rmse_1:.4f}")
print(f"Tempo: {elapsed:.2f} s")

In [0]:
# ------------------------------------------------------------
# Experimento HGB: 4 configurações de hiperparâmetros
# Varia learning_rate, max_iter, max_leaf_nodes e regularização
# ------------------------------------------------------------
hgb_candidates = [
    {
        "name": "HGB_1",
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 0.0
    },
    {
        "name": "HGB_2",
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0
    },
    {
        "name": "HGB_3",
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0
    },
    {
        "name": "HGB_4",
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 30,
        "l2_regularization": 1.0
    }
]

In [0]:
# Loop de treinamento e avaliação dos candidatos HGB
hgb_results = []
hgb_models = {}

for config in hgb_candidates:

    print("\n-------------------------------------")
    print(config["name"])
    print("-------------------------------------")

    model = HistGradientBoostingRegressor(
        learning_rate=config["learning_rate"],
        max_iter=config["max_iter"],
        max_leaf_nodes=config["max_leaf_nodes"],
        min_samples_leaf=config["min_samples_leaf"],
        l2_regularization=config["l2_regularization"],
        early_stopping=False,
        random_state=42
    )

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(X_val)

    elapsed = time.time() - start

    mae = mean_absolute_error(
        y_val,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            prediction
        )
    )

    hgb_results.append({
        "Modelo": config["name"],
        "learning_rate": config["learning_rate"],
        "max_iter": config["max_iter"],
        "max_leaf_nodes": config["max_leaf_nodes"],
        "min_samples_leaf": config["min_samples_leaf"],
        "l2": config["l2_regularization"],
        "MAE": mae,
        "RMSE": rmse,
        "Tempo_segundos": elapsed
    })

    hgb_models[config["name"]] = model

    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Tempo: {elapsed:.2f}s")

In [0]:
# Tabela comparativa dos modelos HGB ordenada por RMSE
hgb_comparison = (
    pd.DataFrame(hgb_results)
    .sort_values("RMSE")
)

display(hgb_comparison)

In [0]:
# Comparação geral: baselines + SARIMA + HGB na validação
comparison_models = [
    {
        "Modelo": "Baseline sazonal 168h",
        "MAE": 0.5275,
        "RMSE": 0.7970
    },
    {
        "Modelo": "SARIMA (1,0,1)(1,0,0,24)",
        "MAE": 0.7051,
        "RMSE": 0.8465
    }
]

for result in hgb_results:
    comparison_models.append({
        "Modelo": result["Modelo"],
        "MAE": result["MAE"],
        "RMSE": result["RMSE"]
    })

validation_comparison = (
    pd.DataFrame(comparison_models)
    .sort_values("RMSE")
)

display(validation_comparison)

In [0]:
# [OBSOLETO] Versão preliminar da função rolling — substituída pela célula 71
# import pandas as pd
#     Os parâmetros do SARIMA NÃO são reestimados.
#     """
#
#     current_result = fitted_result
#
#     predictions = []
#
#     for timestamp, actual_value in validation_series.items():
#
#         # Previsão exatamente 1 passo à frente
#         forecast = current_result.get_forecast(
#             steps=1
#         ).predicted_mean.iloc[0]
#
#         predictions.append({
#             "hour_ts": timestamp,
#             "prediction": forecast
#         })
#
#         # Incorporar a observação real ao estado do modelo
#         new_observation = pd.Series(
#             [actual_value],
#             index=pd.DatetimeIndex(
#                 [timestamp],
#                 freq="h"
#             )
#         )
#
#         current_result = current_result.extend(
#             new_observation
#         )
#
#     return pd.DataFrame(predictions)

In [0]:
# [OBSOLETO] Tentativa de ajustar SARIMA após restart — falhou por perda de variáveis
# from statsmodels.tsa.statespace.sarimax import SARIMAX
#
# sarima_1 = SARIMAX(
#     train_series,
#     order=(1, 0, 1),
#     seasonal_order=(1, 0, 0, 24),
#     trend="c",
#     enforce_stationarity=False,
#     enforce_invertibility=False
# )
#
# sarima_1_result = sarima_1.fit(
#     disp=False,
#     maxiter=50
# )
#
# print(
#     "Convergiu:",
#     sarima_1_result.mle_retvals.get("converged")
# )

In [0]:
# ============================================================
# RECONSTRUIR VARIÁVEIS NECESSÁRIAS PARA O SARIMA
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import numpy as np

# 1. Recuperar os limites cronológicos de treino e validação

train_period = train_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
).first()

validation_period = validation_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
).first()

train_start = train_period["inicio"]
train_end_ts = train_period["fim"]

validation_start = validation_period["inicio"]
validation_end_ts = validation_period["fim"]

print("Período de treino:")
print(train_start, "até", train_end_ts)

print("\nPeríodo de validação:")
print(validation_start, "até", validation_end_ts)


# 2. Recuperar a série horária CONTÍNUA para o SARIMA

sarima_train_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(train_start)) &
        (F.col("hour_ts") <= F.lit(train_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)

sarima_validation_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(validation_start)) &
        (F.col("hour_ts") <= F.lit(validation_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)


# 3. Converter para Pandas

train_pd = sarima_train_spark.toPandas()

validation_calendar_pd = (
    sarima_validation_spark
    .toPandas()
)


# 4. Garantir datetime

train_pd["hour_ts"] = pd.to_datetime(
    train_pd["hour_ts"]
)

validation_calendar_pd["hour_ts"] = pd.to_datetime(
    validation_calendar_pd["hour_ts"]
)


# 5. Criar as séries horárias contínuas

train_series = (
    train_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

validation_series = (
    validation_calendar_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)


# 6. Dataset de avaliação:
# exatamente as mesmas observações válidas usadas
# no baseline e no HGB

validation_eval_pd = (
    validation_df
    .select(
        "hour_ts",
        "power"
    )
    .orderBy("hour_ts")
    .toPandas()
)

validation_eval_pd["hour_ts"] = pd.to_datetime(
    validation_eval_pd["hour_ts"]
)


# 7. Conferência

print("\n====================================")
print("VARIÁVEIS DO SARIMA RECONSTRUÍDAS")
print("====================================")

print(f"Horas calendário no treino: {len(train_series):,}")
print(
    f"Valores ausentes no treino: "
    f"{train_series.isna().sum():,}"
)

print(
    f"\nHoras calendário na validação: "
    f"{len(validation_series):,}"
)

print(
    f"Valores ausentes na validação: "
    f"{validation_series.isna().sum():,}"
)

print(
    f"\nObservações válidas para avaliação: "
    f"{len(validation_eval_pd):,}"
)

In [0]:
# ============================================================
# 00 - RESTAURAÇÃO COMPLETA DO AMBIENTE DO TCC
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import math

# ------------------------------------------------------------
# 1. CARREGAR O ARQUIVO ORIGINAL
# ------------------------------------------------------------

file_path = "/Volumes/workspace/default/tcc/household_power_consumption.txt"

df_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("nullValue", "?")
    .option("inferSchema", False)
    .csv(file_path)
)

# ------------------------------------------------------------
# 2. CONVERTER DATA/HORA E VARIÁVEIS NUMÉRICAS
# ------------------------------------------------------------

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df = df_raw.withColumn(
    "datetime",
    F.to_timestamp(
        F.concat_ws(" ", F.col("Date"), F.col("Time")),
        "d/M/yyyy H:mm:ss"
    )
)

for c in numeric_cols:
    df = df.withColumn(
        c,
        F.col(c).cast(DoubleType())
    )

# ------------------------------------------------------------
# 3. AGREGAÇÃO HORÁRIA
# ------------------------------------------------------------

df_hour_base = (
    df
    .filter(F.col("datetime").isNotNull())
    .withColumn(
        "hour_ts",
        F.date_trunc("hour", F.col("datetime"))
    )
)

hourly_raw = (
    df_hour_base
    .groupBy("hour_ts")
    .agg(
        F.count("Global_active_power").alias("valid_minutes"),
        F.avg("Global_active_power").alias(
            "global_active_power_mean"
        )
    )
)

# ------------------------------------------------------------
# 4. CALENDÁRIO HORÁRIO COMPLETO
# ------------------------------------------------------------

bounds = df.agg(
    F.min("datetime").alias("min_dt"),
    F.max("datetime").alias("max_dt")
)

hour_calendar = bounds.select(
    F.explode(
        F.sequence(
            F.date_trunc("hour", F.col("min_dt")),
            F.date_trunc("hour", F.col("max_dt")),
            F.expr("INTERVAL 1 HOUR")
        )
    ).alias("hour_ts")
)

hourly_complete = (
    hour_calendar
    .join(
        hourly_raw,
        on="hour_ts",
        how="left"
    )
    .withColumn(
        "valid_minutes",
        F.coalesce(
            F.col("valid_minutes"),
            F.lit(0)
        )
    )
    .withColumn(
        "valid_hour",
        F.col("valid_minutes") >= 54
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 5. SÉRIE OFICIAL
# Horas inválidas permanecem, mas power fica NULL
# ------------------------------------------------------------

hourly_model_base = (
    hourly_complete
    .withColumn(
        "power",
        F.when(
            F.col("valid_hour"),
            F.col("global_active_power_mean")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
    .select(
        "hour_ts",
        "power",
        "valid_minutes",
        "valid_hour"
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 6. LAGS
# ------------------------------------------------------------

w = Window.orderBy("hour_ts")

df_features = (
    hourly_model_base
    .withColumn(
        "lag_1",
        F.lag("power", 1).over(w)
    )
    .withColumn(
        "lag_24",
        F.lag("power", 24).over(w)
    )
    .withColumn(
        "lag_168",
        F.lag("power", 168).over(w)
    )
)

# ------------------------------------------------------------
# 7. ATRIBUTOS TEMPORAIS
# ------------------------------------------------------------

df_features = (
    df_features
    .withColumn(
        "hour_of_day",
        F.hour("hour_ts")
    )
    .withColumn(
        "day_of_week",
        F.pmod(
            F.dayofweek("hour_ts") + F.lit(5),
            F.lit(7)
        )
    )
    .withColumn(
        "month",
        F.month("hour_ts")
    )
)

# ------------------------------------------------------------
# 8. ATRIBUTOS CÍCLICOS
# ------------------------------------------------------------

df_features = (
    df_features
    .withColumn(
        "hour_sin",
        F.sin(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "hour_cos",
        F.cos(
            F.col("hour_of_day") *
            F.lit(2 * math.pi / 24)
        )
    )
    .withColumn(
        "dow_sin",
        F.sin(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
    .withColumn(
        "dow_cos",
        F.cos(
            F.col("day_of_week") *
            F.lit(2 * math.pi / 7)
        )
    )
    .withColumn(
        "month_sin",
        F.sin(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
    .withColumn(
        "month_cos",
        F.cos(
            (F.col("month") - 1) *
            F.lit(2 * math.pi / 12)
        )
    )
)

# ------------------------------------------------------------
# 9. MÉDIAS MÓVEIS SEM DATA LEAKAGE
# ------------------------------------------------------------

w_roll_24 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-24, -1)
)

w_roll_168 = (
    Window
    .orderBy("hour_ts")
    .rowsBetween(-168, -1)
)

df_features = (
    df_features
    .withColumn(
        "rolling_count_24",
        F.count("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24_raw",
        F.avg("power").over(w_roll_24)
    )
    .withColumn(
        "rolling_mean_24",
        F.when(
            F.col("rolling_count_24") == 24,
            F.col("rolling_mean_24_raw")
        )
    )
    .withColumn(
        "rolling_count_168",
        F.count("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168_raw",
        F.avg("power").over(w_roll_168)
    )
    .withColumn(
        "rolling_mean_168",
        F.when(
            F.col("rolling_count_168") == 168,
            F.col("rolling_mean_168_raw")
        )
    )
)

# ------------------------------------------------------------
# 10. DATASET PRINCIPAL
# ------------------------------------------------------------

model_core = (
    df_features
    .filter(
        F.col("power").isNotNull() &
        F.col("lag_1").isNotNull() &
        F.col("lag_24").isNotNull() &
        F.col("lag_168").isNotNull()
    )
    .orderBy("hour_ts")
)

# ------------------------------------------------------------
# 11. DIVISÃO CRONOLÓGICA
# 64% treino / 16% validação / 20% teste
# ------------------------------------------------------------

w_order = Window.orderBy("hour_ts")

model_indexed = (
    model_core
    .withColumn(
        "row_number",
        F.row_number().over(w_order)
    )
)

total_model = model_indexed.count()

train_end = int(total_model * 0.64)
validation_end = int(total_model * 0.80)

train_df = (
    model_indexed
    .filter(
        F.col("row_number") <= train_end
    )
)

validation_df = (
    model_indexed
    .filter(
        (F.col("row_number") > train_end) &
        (F.col("row_number") <= validation_end)
    )
)

test_df = (
    model_indexed
    .filter(
        F.col("row_number") > validation_end
    )
)

# ------------------------------------------------------------
# 12. LIMITES TEMPORAIS PARA O SARIMA
# ------------------------------------------------------------

train_period = train_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
).first()

validation_period = validation_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
).first()

train_start = train_period["inicio"]
train_end_ts = train_period["fim"]

validation_start = validation_period["inicio"]
validation_end_ts = validation_period["fim"]

# ------------------------------------------------------------
# 13. SÉRIES CONTÍNUAS PARA O SARIMA
# ------------------------------------------------------------

sarima_train_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(train_start)) &
        (F.col("hour_ts") <= F.lit(train_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)

sarima_validation_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(validation_start)) &
        (F.col("hour_ts") <= F.lit(validation_end_ts))
    )
    .select("hour_ts", "power")
    .orderBy("hour_ts")
)

train_pd = sarima_train_spark.toPandas()
validation_calendar_pd = sarima_validation_spark.toPandas()

train_pd["hour_ts"] = pd.to_datetime(
    train_pd["hour_ts"]
)

validation_calendar_pd["hour_ts"] = pd.to_datetime(
    validation_calendar_pd["hour_ts"]
)

train_series = (
    train_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

validation_series = (
    validation_calendar_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

# ------------------------------------------------------------
# 14. OBSERVAÇÕES VÁLIDAS PARA AVALIAÇÃO DO SARIMA
# ------------------------------------------------------------

validation_eval_pd = (
    validation_df
    .select(
        "hour_ts",
        "power"
    )
    .orderBy("hour_ts")
    .toPandas()
)

validation_eval_pd["hour_ts"] = pd.to_datetime(
    validation_eval_pd["hour_ts"]
)

# ------------------------------------------------------------
# 15. CONFERÊNCIA FINAL
# ------------------------------------------------------------

print("==========================================")
print("AMBIENTE RESTAURADO COM SUCESSO")
print("==========================================")

print(f"Dataset original: {df_raw.count():,}")
print(f"Horas calendário: {hourly_complete.count():,}")
print(f"Model core: {total_model:,}")

print()
print(f"Treino:    {train_df.count():,}")
print(f"Validação: {validation_df.count():,}")
print(f"Teste:     {test_df.count():,}")

print()
print("Período treino:")
print(train_start, "até", train_end_ts)

print("\nPeríodo validação:")
print(validation_start, "até", validation_end_ts)

print()
print(f"Horas contínuas treino SARIMA: {len(train_series):,}")
print(f"NaN treino SARIMA: {train_series.isna().sum():,}")

print(
    f"Horas contínuas validação SARIMA: "
    f"{len(validation_series):,}"
)

print(
    f"NaN validação SARIMA: "
    f"{validation_series.isna().sum():,}"
)

print(
    f"Observações válidas para avaliação: "
    f"{len(validation_eval_pd):,}"
)

In [0]:
# Verificação rápida dos tipos das séries após a restauração
print(type(train_series))
print(type(validation_series))

print(train_series.head())

In [0]:
# ------------------------------------------------------------
# Função rolling one-step: a cada hora prevê 1 passo à frente,
# incorpora a observação real e avança — parâmetros fixos
# ------------------------------------------------------------
import pandas as pd

def rolling_one_step_forecast(fitted_result, validation_series):
    """
    Previsão rolling de 1 hora à frente.

    A cada instante:
    1. prevê a próxima hora;
    2. incorpora o valor real observado;
    3. avança para a hora seguinte.

    Os parâmetros do SARIMA permanecem fixos.
    """

    current_result = fitted_result
    predictions = []

    for timestamp, actual_value in validation_series.items():

        # previsão exatamente 1 passo à frente
        forecast = current_result.get_forecast(
            steps=1
        ).predicted_mean.iloc[0]

        predictions.append({
            "hour_ts": timestamp,
            "prediction": forecast
        })

        # incorporar a observação real sem reestimar parâmetros
        current_result = current_result.extend(
            [actual_value]
        )

    return pd.DataFrame(predictions)

print("Função criada com sucesso.")

In [0]:
# ------------------------------------------------------------
# SARIMA 1 rolling: ajusta na série de treino e faz previsão
# recursiva de 1h à frente sobre a validação
# ------------------------------------------------------------
from statsmodels.tsa.statespace.sarimax import SARIMAX

start = time.time()

sarima_1 = SARIMAX(
    train_series,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 0, 24),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_1_result = sarima_1.fit(
    disp=False,
    maxiter=50
)

fit_time_1 = time.time() - start

print("SARIMA 1")
print(
    "Convergiu:",
    sarima_1_result.mle_retvals.get("converged")
)
print(f"Tempo de ajuste: {fit_time_1:.1f}s")

In [0]:
# Executa rolling one-step para SARIMA 1
start = time.time()

rolling_1_pd = rolling_one_step_forecast(
    sarima_1_result,
    validation_series
)

rolling_time_1 = time.time() - start

print(
    f"Previsões geradas: {len(rolling_1_pd):,}"
)
print(
    f"Tempo rolling: {rolling_time_1:.1f}s"
)

In [0]:
# Avaliação do SARIMA 1 rolling na validação (apenas horas com target)
eval_rolling_1 = validation_eval_pd.merge(
    rolling_1_pd,
    on="hour_ts",
    how="inner"
)

# remover qualquer previsão NaN, se houver
eval_rolling_1 = eval_rolling_1.dropna(
    subset=["power", "prediction"]
)

mae_rolling_1 = np.mean(
    np.abs(
        eval_rolling_1["power"] -
        eval_rolling_1["prediction"]
    )
)

rmse_rolling_1 = np.sqrt(
    np.mean(
        (
            eval_rolling_1["power"] -
            eval_rolling_1["prediction"]
        ) ** 2
    )
)

print("SARIMA 1 — ROLLING ONE-STEP")
print(f"MAE:  {mae_rolling_1:.4f}")
print(f"RMSE: {rmse_rolling_1:.4f}")
print(f"N:    {len(eval_rolling_1):,}")

In [0]:
# ------------------------------------------------------------
# SARIMA 3 rolling: (1,0,1)(1,1,0,24) com diferenciação sazonal
# ------------------------------------------------------------
start = time.time()

sarima_3 = SARIMAX(
    train_series,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 0, 24),
    trend=None,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_3_result = sarima_3.fit(
    disp=False,
    maxiter=50
)

fit_time_3 = time.time() - start

print("SARIMA 3")
print(
    "Convergiu:",
    sarima_3_result.mle_retvals.get("converged")
)
print(f"Tempo de ajuste: {fit_time_3:.1f}s")

In [0]:
# Executa rolling one-step para SARIMA 3
start = time.time()

rolling_3_pd = rolling_one_step_forecast(
    sarima_3_result,
    validation_series
)

rolling_time_3 = time.time() - start

print(
    f"Previsões geradas: {len(rolling_3_pd):,}"
)
print(
    f"Tempo rolling: {rolling_time_3:.1f}s"
)

In [0]:
eval_rolling_3 = validation_eval_pd.merge(
    rolling_3_pd,
    on="hour_ts",
    how="inner"
)

eval_rolling_3 = eval_rolling_3.dropna(
    subset=["power", "prediction"]
)

mae_rolling_3 = np.mean(
    np.abs(
        eval_rolling_3["power"] -
        eval_rolling_3["prediction"]
    )
)

rmse_rolling_3 = np.sqrt(
    np.mean(
        (
            eval_rolling_3["power"] -
            eval_rolling_3["prediction"]
        ) ** 2
    )
)

print("SARIMA 3 — ROLLING ONE-STEP")
print(f"MAE:  {mae_rolling_3:.4f}")
print(f"RMSE: {rmse_rolling_3:.4f}")
print(f"N:    {len(eval_rolling_3):,}")

In [0]:
# Tabela comparativa final na validação: Baseline vs SARIMA rolling vs HGB
correct_comparison = pd.DataFrame([
    {
        "Modelo": "Baseline sazonal 168h",
        "MAE": 0.5275,
        "RMSE": 0.7970
    },
    {
        "Modelo": "SARIMA 1 rolling",
        "MAE": mae_rolling_1,
        "RMSE": rmse_rolling_1
    },
    {
        "Modelo": "SARIMA 3 rolling",
        "MAE": mae_rolling_3,
        "RMSE": rmse_rolling_3
    },
    {
        "Modelo": "HGB_3",
        "MAE": 0.3422,
        "RMSE": 0.4988
    }
]).sort_values("RMSE")

display(correct_comparison)

## Etapa 6 — Avaliação Final no Conjunto de Teste

Após a etapa de validação, foram selecionadas e congeladas as seguintes configurações:

- Baseline sazonal: valor da mesma hora da semana anterior (lag 168h);
- SARIMA: (1,0,1)(1,0,0,24), avaliado em rolling one-step;
- HistGradientBoosting:
  - learning_rate = 0.03
  - max_iter = 300
  - max_leaf_nodes = 15
  - min_samples_leaf = 20
  - l2_regularization = 1.0

O conjunto de teste não foi utilizado para seleção de modelos ou hiperparâmetros.
Após sua abertura, nenhuma configuração será alterada em função dos resultados obtidos.

In [0]:
# ------------------------------------------------------------
# Etapa 6 — Consolidação do conjunto de desenvolvimento
# (treino + validação) para retreinar modelos antes do teste
# ------------------------------------------------------------
development_df = (
    train_df
    .unionByName(validation_df)
    .orderBy("hour_ts")
)

print(f"Desenvolvimento: {development_df.count():,}")
print(f"Teste: {test_df.count():,}")

In [0]:
# Exibe período coberto por desenvolvimento e teste
development_period = development_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
)

test_period = test_df.agg(
    F.min("hour_ts").alias("inicio"),
    F.max("hour_ts").alias("fim")
)

print("DESENVOLVIMENTO")
display(development_period)

print("TESTE")
display(test_period)

In [0]:
# ------------------------------------------------------------
# Baseline sazonal 168h no teste: usa lag_168 como previsão
# ------------------------------------------------------------
test_baseline_pd = (
    test_df
    .select(
        "hour_ts",
        "power",
        "lag_168"
    )
    .orderBy("hour_ts")
    .toPandas()
)

baseline_test_mae = mean_absolute_error(
    test_baseline_pd["power"],
    test_baseline_pd["lag_168"]
)

baseline_test_rmse = np.sqrt(
    mean_squared_error(
        test_baseline_pd["power"],
        test_baseline_pd["lag_168"]
    )
)

print("BASELINE SAZONAL 168h — TESTE")
print(f"MAE:  {baseline_test_mae:.4f}")
print(f"RMSE: {baseline_test_rmse:.4f}")
print(f"N:    {len(test_baseline_pd):,}")

In [0]:
# Importa métricas do sklearn para avaliação no teste
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

print("Métricas importadas com sucesso.")

In [0]:
# Avaliação do baseline 168h no conjunto de teste
baseline_test_mae = mean_absolute_error(
    test_baseline_pd["power"],
    test_baseline_pd["lag_168"]
)

baseline_test_rmse = np.sqrt(
    mean_squared_error(
        test_baseline_pd["power"],
        test_baseline_pd["lag_168"]
    )
)

print("BASELINE SAZONAL 168h — TESTE")
print(f"MAE:  {baseline_test_mae:.4f}")
print(f"RMSE: {baseline_test_rmse:.4f}")
print(f"N:    {len(test_baseline_pd):,}")

In [0]:
# Definição das features e target para HGB no teste
feature_cols = [
    "lag_1",
    "lag_24",
    "lag_168",

    "rolling_mean_24",
    "rolling_mean_168",

    "hour_sin",
    "hour_cos",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos"
]

target_col = "power"

In [0]:
# Converte desenvolvimento e teste para pandas com features
development_ml_pd = (
    development_df
    .select(
        "hour_ts",
        target_col,
        *feature_cols
    )
    .orderBy("hour_ts")
    .toPandas()
)

test_ml_pd = (
    test_df
    .select(
        "hour_ts",
        target_col,
        *feature_cols
    )
    .orderBy("hour_ts")
    .toPandas()
)

print(
    f"Desenvolvimento ML: "
    f"{len(development_ml_pd):,}"
)

print(
    f"Teste ML: "
    f"{len(test_ml_pd):,}"
)

In [0]:
# Separa X e y para desenvolvimento (treino+val) e teste
X_dev = development_ml_pd[feature_cols]
y_dev = development_ml_pd[target_col]

X_test = test_ml_pd[feature_cols]
y_test = test_ml_pd[target_col]

In [0]:
# ------------------------------------------------------------
# HGB final: retreina com desenvolvimento (treino+validação)
# Hiperparâmetros congelados da etapa de validação
# ------------------------------------------------------------
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

hgb_final = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=300,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

start = time.time()

hgb_final.fit(
    X_dev,
    y_dev
)

hgb_fit_time = time.time() - start

print(
    f"Treinamento concluído em "
    f"{hgb_fit_time:.2f}s"
)

In [0]:
# Previsão e avaliação do HGB no teste
hgb_test_pred = hgb_final.predict(
    X_test
)

hgb_test_mae = mean_absolute_error(
    y_test,
    hgb_test_pred
)

hgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        hgb_test_pred
    )
)

print("HISTGRADIENTBOOSTING — TESTE")
print(f"MAE:  {hgb_test_mae:.4f}")
print(f"RMSE: {hgb_test_rmse:.4f}")
print(f"N:    {len(y_test):,}")

In [0]:
# Extrai limites temporais para filtrar séries do SARIMA
development_start = (
    development_df
    .agg(F.min("hour_ts"))
    .first()[0]
)

development_end = (
    development_df
    .agg(F.max("hour_ts"))
    .first()[0]
)

test_start = (
    test_df
    .agg(F.min("hour_ts"))
    .first()[0]
)

test_end = (
    test_df
    .agg(F.max("hour_ts"))
    .first()[0]
)

print(
    "Desenvolvimento:",
    development_start,
    "até",
    development_end
)

print(
    "Teste:",
    test_start,
    "até",
    test_end
)

In [0]:
# Filtra séries horárias contínuas para SARIMA (desenvolvimento e teste)
sarima_development_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(development_start)) &
        (F.col("hour_ts") <= F.lit(development_end))
    )
    .select(
        "hour_ts",
        "power"
    )
    .orderBy("hour_ts")
)

sarima_test_spark = (
    hourly_model_base
    .filter(
        (F.col("hour_ts") >= F.lit(test_start)) &
        (F.col("hour_ts") <= F.lit(test_end))
    )
    .select(
        "hour_ts",
        "power"
    )
    .orderBy("hour_ts")
)

In [0]:
# Converte séries do SARIMA para pandas e define frequência horária
sarima_dev_pd = (
    sarima_development_spark
    .toPandas()
)

sarima_test_calendar_pd = (
    sarima_test_spark
    .toPandas()
)

sarima_dev_pd["hour_ts"] = pd.to_datetime(
    sarima_dev_pd["hour_ts"]
)

sarima_test_calendar_pd["hour_ts"] = pd.to_datetime(
    sarima_test_calendar_pd["hour_ts"]
)

development_series = (
    sarima_dev_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

test_series = (
    sarima_test_calendar_pd
    .set_index("hour_ts")["power"]
    .asfreq("h")
)

In [0]:
# Conferência das séries do SARIMA: tamanho e valores ausentes
print(
    "Horas calendário desenvolvimento:",
    len(development_series)
)

print(
    "NaN desenvolvimento:",
    development_series.isna().sum()
)

print(
    "Horas calendário teste:",
    len(test_series)
)

print(
    "NaN calendário teste:",
    test_series.isna().sum()
)

In [0]:
# ------------------------------------------------------------
# SARIMA final: ajusta com desenvolvimento (treino+validação)
# Configuração congelada: (1,0,1)(1,0,0,24)
# ------------------------------------------------------------
from statsmodels.tsa.statespace.sarimax import SARIMAX

start = time.time()

sarima_final = SARIMAX(
    development_series,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 0, 24),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_final_result = sarima_final.fit(
    disp=False,
    maxiter=50
)

sarima_fit_time = time.time() - start

print(
    "Convergiu:",
    sarima_final_result.mle_retvals.get(
        "converged"
    )
)

print(
    f"Tempo de ajuste: "
    f"{sarima_fit_time:.1f}s"
)

In [0]:
# Rolling one-step SARIMA no teste
start = time.time()

sarima_test_pred_pd = rolling_one_step_forecast(
    sarima_final_result,
    test_series
)

sarima_rolling_time = time.time() - start

print(
    f"Previsões calendário: "
    f"{len(sarima_test_pred_pd):,}"
)

print(
    f"Tempo rolling: "
    f"{sarima_rolling_time:.1f}s"
)

In [0]:
# DataFrame de avaliação: horas com target válido no teste
test_eval_pd = (
    test_df
    .select(
        "hour_ts",
        "power"
    )
    .orderBy("hour_ts")
    .toPandas()
)

test_eval_pd["hour_ts"] = pd.to_datetime(
    test_eval_pd["hour_ts"]
)

In [0]:
# Avaliação do SARIMA rolling no teste (apenas horas com target)
sarima_test_eval = (
    test_eval_pd
    .merge(
        sarima_test_pred_pd,
        on="hour_ts",
        how="inner"
    )
    .dropna(
        subset=[
            "power",
            "prediction"
        ]
    )
)

sarima_test_mae = mean_absolute_error(
    sarima_test_eval["power"],
    sarima_test_eval["prediction"]
)

sarima_test_rmse = np.sqrt(
    mean_squared_error(
        sarima_test_eval["power"],
        sarima_test_eval["prediction"]
    )
)

print("SARIMA ROLLING — TESTE")
print(f"MAE:  {sarima_test_mae:.4f}")
print(f"RMSE: {sarima_test_rmse:.4f}")
print(f"N:    {len(sarima_test_eval):,}")

In [0]:
# ------------------------------------------------------------
# Tabela final de resultados no teste: Baseline vs SARIMA vs HGB
# ------------------------------------------------------------
final_test_results = pd.DataFrame([
    {
        "Modelo": "Baseline sazonal 168h",
        "MAE": baseline_test_mae,
        "RMSE": baseline_test_rmse
    },
    {
        "Modelo": "SARIMA (1,0,1)(1,0,0,24)",
        "MAE": sarima_test_mae,
        "RMSE": sarima_test_rmse
    },
    {
        "Modelo": "HistGradientBoosting",
        "MAE": hgb_test_mae,
        "RMSE": hgb_test_rmse
    }
]).sort_values("RMSE")

display(final_test_results)

In [0]:
# ------------------------------------------------------------
# Junção das previsões de todos os modelos em um único DataFrame
# Colunas: real, baseline_168, sarima, hgb
# ------------------------------------------------------------
import pandas as pd

baseline_final_pd = (
    test_baseline_pd[
        ["hour_ts", "power", "lag_168"]
    ]
    .copy()
)

baseline_final_pd = baseline_final_pd.rename(
    columns={
        "power": "real",
        "lag_168": "baseline_168"
    }
)

baseline_final_pd["hour_ts"] = pd.to_datetime(
    baseline_final_pd["hour_ts"]
)

# ------------------------------------------------------------
# SARIMA
# ------------------------------------------------------------

sarima_final_pd = (
    sarima_test_eval[
        ["hour_ts", "prediction"]
    ]
    .copy()
)

sarima_final_pd = sarima_final_pd.rename(
    columns={
        "prediction": "sarima"
    }
)

sarima_final_pd["hour_ts"] = pd.to_datetime(
    sarima_final_pd["hour_ts"]
)

# ------------------------------------------------------------
# HISTGRADIENTBOOSTING
# ------------------------------------------------------------

hgb_final_pd = pd.DataFrame({
    "hour_ts": pd.to_datetime(
        test_ml_pd["hour_ts"]
    ),
    "hgb": hgb_test_pred
})

# ------------------------------------------------------------
# JUNÇÃO FINAL
# ------------------------------------------------------------

final_predictions = (
    baseline_final_pd
    .merge(
        sarima_final_pd,
        on="hour_ts",
        how="inner"
    )
    .merge(
        hgb_final_pd,
        on="hour_ts",
        how="inner"
    )
    .sort_values("hour_ts")
    .reset_index(drop=True)
)

print(final_predictions.head())

print(
    "\nNúmero de observações:",
    len(final_predictions)
)

In [0]:
# Calcula erros absolutos e diretos para cada modelo
final_predictions["erro_baseline"] = (
    final_predictions["real"] -
    final_predictions["baseline_168"]
)

final_predictions["erro_sarima"] = (
    final_predictions["real"] -
    final_predictions["sarima"]
)

final_predictions["erro_hgb"] = (
    final_predictions["real"] -
    final_predictions["hgb"]
)

final_predictions["erro_abs_baseline"] = (
    np.abs(final_predictions["erro_baseline"])
)

final_predictions["erro_abs_sarima"] = (
    np.abs(final_predictions["erro_sarima"])
)

final_predictions["erro_abs_hgb"] = (
    np.abs(final_predictions["erro_hgb"])
)

display(final_predictions.head())

In [0]:
# Cria diretório de resultados no Volume UC
import os

results_path = (
    "/Volumes/workspace/default/tcc/resultados"
)

os.makedirs(
    results_path,
    exist_ok=True
)

print(results_path)

In [0]:
# Salva previsões em CSV
csv_path = (
    f"{results_path}/previsoes_teste_final.csv"
)

final_predictions.to_csv(
    csv_path,
    index=False
)

print(
    "CSV salvo em:",
    csv_path
)

In [0]:
# Salva previsões em formato Delta (para consultas SQL)
final_predictions_spark = (
    spark.createDataFrame(
        final_predictions
    )
)

delta_path = (
    f"{results_path}/previsoes_teste_delta"
)

(
    final_predictions_spark
    .write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

print(
    "Delta salvo em:",
    delta_path
)

In [0]:
# Recarrega previsões salvas em formato Delta para análise
final_predictions_spark = (
    spark.read
    .format("delta")
    .load(
        "/Volumes/workspace/default/tcc/resultados/previsoes_teste_delta"
    )
)

In [0]:
# Salva o dataset principal (model_core) em Delta para reuso
model_core_path = (
    f"{results_path}/model_core_delta"
)

(
    model_core
    .write
    .format("delta")
    .mode("overwrite")
    .save(model_core_path)
)

print(
    "Model core salvo em:",
    model_core_path
)

In [0]:
# ------------------------------------------------------------
# Gráfico de barras: MAE e RMSE dos modelos no teste
# ------------------------------------------------------------
import matplotlib.pyplot as plt

metrics_final = pd.DataFrame({
    "Modelo": [
        "Baseline 168h",
        "SARIMA",
        "HistGradientBoosting"
    ],
    "MAE": [
        0.5682,
        0.3902,
        0.3347
    ],
    "RMSE": [
        0.8197,
        0.5363,
        0.4810
    ]
})

ax = metrics_final.set_index(
    "Modelo"
)[["MAE", "RMSE"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Desempenho dos modelos no conjunto de teste"
)

plt.ylabel(
    "Erro (kW)"
)

plt.xlabel(
    "Modelo"
)

plt.xticks(
    rotation=0
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/grafico_mae_rmse.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Gráfico de linha: potência real vs prevista nos primeiros 7 dias do teste
plot_window = (
    final_predictions
    .iloc[:24 * 7]
    .copy()
)

plt.figure(
    figsize=(15, 6)
)

plt.plot(
    plot_window["hour_ts"],
    plot_window["real"],
    label="Real",
    linewidth=2
)

plt.plot(
    plot_window["hour_ts"],
    plot_window["baseline_168"],
    label="Baseline 168h",
    alpha=0.8
)

plt.plot(
    plot_window["hour_ts"],
    plot_window["sarima"],
    label="SARIMA",
    alpha=0.8
)

plt.plot(
    plot_window["hour_ts"],
    plot_window["hgb"],
    label="HistGradientBoosting",
    alpha=0.8
)

plt.title(
    "Potência ativa real e prevista — exemplo de 7 dias"
)

plt.xlabel(
    "Data"
)

plt.ylabel(
    "Potência ativa média (kW)"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/grafico_real_previsoes_7dias.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Gráfico de dispersão: valores reais x previstos (HGB)
plt.scatter(
    final_predictions["real"],
    final_predictions["hgb"],
    alpha=0.25,
    s=12
)

min_value = min(
    final_predictions["real"].min(),
    final_predictions["hgb"].min()
)

max_value = max(
    final_predictions["real"].max(),
    final_predictions["hgb"].max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title(
    "Valores reais × previstos — HistGradientBoosting"
)

plt.xlabel(
    "Potência real (kW)"
)

plt.ylabel(
    "Potência prevista (kW)"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/grafico_real_vs_hgb.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Histograma: distribuição dos erros de previsão por modelo
plt.hist(
    final_predictions["erro_baseline"],
    bins=60,
    alpha=0.5,
    label="Baseline"
)

plt.hist(
    final_predictions["erro_sarima"],
    bins=60,
    alpha=0.5,
    label="SARIMA"
)

plt.hist(
    final_predictions["erro_hgb"],
    bins=60,
    alpha=0.5,
    label="HistGradientBoosting"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.title(
    "Distribuição dos erros de previsão"
)

plt.xlabel(
    "Erro: valor real − valor previsto (kW)"
)

plt.ylabel(
    "Frequência"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    f"{results_path}/grafico_distribuicao_erros.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Tabela resumo de erros: média, desvio, mediana e máximo
error_summary = pd.DataFrame({
    "Modelo": [
        "Baseline 168h",
        "SARIMA",
        "HistGradientBoosting"
    ],
    "Erro médio": [
        final_predictions["erro_baseline"].mean(),
        final_predictions["erro_sarima"].mean(),
        final_predictions["erro_hgb"].mean()
    ],
    "Desvio padrão do erro": [
        final_predictions["erro_baseline"].std(),
        final_predictions["erro_sarima"].std(),
        final_predictions["erro_hgb"].std()
    ],
    "Erro absoluto mediano": [
        final_predictions["erro_abs_baseline"].median(),
        final_predictions["erro_abs_sarima"].median(),
        final_predictions["erro_abs_hgb"].median()
    ],
    "Erro absoluto máximo": [
        final_predictions["erro_abs_baseline"].max(),
        final_predictions["erro_abs_sarima"].max(),
        final_predictions["erro_abs_hgb"].max()
    ]
})

display(
    error_summary
)

In [0]:
# Top 20 maiores erros absolutos do HGB (anomalias)
worst_hgb = (
    final_predictions
    .sort_values(
        "erro_abs_hgb",
        ascending=False
    )
    .head(20)
)

display(
    worst_hgb[
        [
            "hour_ts",
            "real",
            "hgb",
            "erro_hgb",
            "erro_abs_hgb"
        ]
    ]
)

In [0]:
# Lista arquivos salvos no diretório de resultados
print(os.listdir(results_path))

In [0]:
# Verificação de NaNs nas colunas de erro
print("N:", len(final_predictions))

print(
    final_predictions[
        [
            "erro_baseline",
            "erro_sarima",
            "erro_hgb"
        ]
    ].isna().sum()
)

In [0]:
# ------------------------------------------------------------
# Teste Diebold-Mariano: compara acurácia preditiva entre modelos
# H0: modelos têm mesma acurácia; função de perda: erro quadrático
# ------------------------------------------------------------
import numpy as np
from scipy import stats


def diebold_mariano(
    errors_model_1,
    errors_model_2,
    h=1
):
    """
    Teste Diebold-Mariano para igualdade
    de acurácia preditiva.

    Função de perda:
    erro quadrático.

    d_t < 0:
    modelo 1 apresenta menor perda média.

    d_t > 0:
    modelo 2 apresenta menor perda média.
    """

    e1 = np.asarray(errors_model_1)
    e2 = np.asarray(errors_model_2)

    # diferencial de perdas
    d = (e1 ** 2) - (e2 ** 2)

    d = d[np.isfinite(d)]

    n = len(d)

    mean_d = np.mean(d)

    # variância do diferencial
    gamma_0 = np.mean(
        (d - mean_d) ** 2
    )

    variance = gamma_0

    # Para previsões h passos à frente,
    # incorporar autocovariâncias até h-1
    for lag in range(1, h):

        gamma = np.mean(
            (d[lag:] - mean_d) *
            (d[:-lag] - mean_d)
        )

        variance += 2 * gamma

    dm_stat = (
        mean_d /
        np.sqrt(variance / n)
    )

    # Correção de Harvey-Leybourne-Newbold
    correction = np.sqrt(
        (
            n + 1
            - 2 * h
            + (h * (h - 1) / n)
        ) / n
    )

    dm_stat_corrected = (
        dm_stat * correction
    )

    p_value = 2 * (
        1 -
        stats.t.cdf(
            abs(dm_stat_corrected),
            df=n - 1
        )
    )

    return {
        "DM": dm_stat_corrected,
        "p_value": p_value,
        "mean_loss_difference": mean_d,
        "n": n
    }

In [0]:
# Teste DM: HGB vs SARIMA
dm_hgb_sarima = diebold_mariano(
    final_predictions["erro_hgb"],
    final_predictions["erro_sarima"],
    h=1
)

dm_hgb_sarima

In [0]:
# Teste DM: HGB vs Baseline
dm_hgb_baseline = diebold_mariano(
    final_predictions["erro_hgb"],
    final_predictions["erro_baseline"],
    h=1
)

dm_hgb_baseline

In [0]:
# Teste DM: SARIMA vs Baseline
dm_sarima_baseline = diebold_mariano(
    final_predictions["erro_sarima"],
    final_predictions["erro_baseline"],
    h=1
)

dm_sarima_baseline

In [0]:
# Tabela consolidada dos testes Diebold-Mariano
dm_results = pd.DataFrame([
    {
        "Comparação": "HGB × SARIMA",
        "DM": dm_hgb_sarima["DM"],
        "p-valor": dm_hgb_sarima["p_value"],
        "Diferença média de perda":
            dm_hgb_sarima[
                "mean_loss_difference"
            ]
    },
    {
        "Comparação": "HGB × Baseline",
        "DM": dm_hgb_baseline["DM"],
        "p-valor": dm_hgb_baseline["p_value"],
        "Diferença média de perda":
            dm_hgb_baseline[
                "mean_loss_difference"
            ]
    },
    {
        "Comparação": "SARIMA × Baseline",
        "DM": dm_sarima_baseline["DM"],
        "p-valor": dm_sarima_baseline["p_value"],
        "Diferença média de perda":
            dm_sarima_baseline[
                "mean_loss_difference"
            ]
    }
])

display(dm_results)

In [0]:
# ------------------------------------------------------------
# Análise por faixa de potência: divide o teste em bins de 1 kW
# ------------------------------------------------------------
bins = [
    -np.inf,
    1,
    2,
    3,
    4,
    np.inf
]

labels = [
    "< 1 kW",
    "1–2 kW",
    "2–3 kW",
    "3–4 kW",
    ">= 4 kW"
]

final_predictions["faixa_potencia"] = pd.cut(
    final_predictions["real"],
    bins=bins,
    labels=labels,
    right=False
)

In [0]:
# Calcula MAE por faixa de potência para cada modelo
band_results = []

for faixa in labels:

    subset = final_predictions[
        final_predictions["faixa_potencia"] == faixa
    ]

    band_results.append({
        "Faixa": faixa,
        "N": len(subset),

        "MAE_Baseline": np.mean(
            np.abs(subset["erro_baseline"])
        ),

        "MAE_SARIMA": np.mean(
            np.abs(subset["erro_sarima"])
        ),

        "MAE_HGB": np.mean(
            np.abs(subset["erro_hgb"])
        ),

        "Erro_medio_HGB": np.mean(
            subset["erro_hgb"]
        )
    })

band_summary = pd.DataFrame(
    band_results
)

display(band_summary)

In [0]:
# Adiciona percentual de observações por faixa
band_summary["Percentual"] = (
    band_summary["N"] /
    len(final_predictions) *
    100
)

display(band_summary)

In [0]:
# Tabela final da análise por faixa de potência
display(
    band_summary[
        [
            "Faixa",
            "N",
            "Percentual",
            "MAE_Baseline",
            "MAE_SARIMA",
            "MAE_HGB",
            "Erro_medio_HGB"
        ]
    ]
)

In [0]:
# ------------------------------------------------------------
# Etapa 7 — EDA sobre o conjunto de desenvolvimento (treino+validação)
# Carrega model_core salvo e filtra apenas o período de desenvolvimento
# ------------------------------------------------------------
from pyspark.sql import functions as F
import pandas as pd

model_core_eda = (
    spark.read
    .format("delta")
    .load(
        f"{results_path}/model_core_delta"
    )
)

# O teste começa em 29/01/2010 às 14:00.
# Para a EDA usamos somente treino + validação.

development_eda = (
    model_core_eda
    .filter(
        F.col("hour_ts") < F.lit("2010-01-29 14:00:00")
    )
    .orderBy("hour_ts")
)

print(
    "Observações da EDA:",
    development_eda.count()
)

In [0]:
# Converte para pandas e calcula estatísticas descritivas da potência
eda_pd = (
    development_eda
    .select(
        "hour_ts",
        "power",
        "lag_1",
        "lag_24",
        "lag_168",
        "hour_of_day",
        "day_of_week",
        "month"
    )
    .orderBy("hour_ts")
    .toPandas()
)

eda_pd["hour_ts"] = pd.to_datetime(
    eda_pd["hour_ts"]
)

descriptive_summary = (
    eda_pd["power"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75
        ]
    )
)

print(descriptive_summary)

In [0]:
# Histograma: distribuição da potência ativa média horária
plt.figure(
    figsize=(10, 6)
)

plt.hist(
    eda_pd["power"],
    bins=60
)

plt.title(
    "Distribuição da potência ativa média horária"
)

plt.xlabel(
    "Potência ativa média (kW)"
)

plt.ylabel(
    "Frequência"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/eda_distribuicao_potencia.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Perfil horário: média, mediana e contagem por hora do dia
hourly_profile = (
    eda_pd
    .groupby(
        "hour_of_day",
        as_index=False
    )["power"]
    .agg(
        ["mean", "median", "count"]
    )
    .reset_index()
)

display(hourly_profile)

In [0]:
# Gráfico de linha: potência média e mediana por hora do dia
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    hourly_profile["hour_of_day"],
    hourly_profile["mean"],
    marker="o",
    label="Média"
)

plt.plot(
    hourly_profile["hour_of_day"],
    hourly_profile["median"],
    marker="o",
    label="Mediana"
)

plt.xticks(
    range(0, 24)
)

plt.title(
    "Potência ativa segundo a hora do dia"
)

plt.xlabel(
    "Hora do dia"
)

plt.ylabel(
    "Potência ativa média (kW)"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/eda_perfil_horario.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Top 5 horas com maior potência média
peak_hour = (
    hourly_profile
    .sort_values(
        "mean",
        ascending=False
    )
    .head(5)
)

display(peak_hour)

In [0]:
# ------------------------------------------------------------
# Perfil por dia da semana: média, mediana e contagem
# ------------------------------------------------------------
day_names = {
    0: "Segunda",
    1: "Terça",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sábado",
    6: "Domingo"
}

daily_profile = (
    eda_pd
    .groupby(
        "day_of_week",
        as_index=False
    )["power"]
    .agg(
        ["mean", "median", "count"]
    )
    .reset_index()
)

daily_profile["Dia"] = (
    daily_profile["day_of_week"]
    .map(day_names)
)

display(daily_profile)

In [0]:
# Gráfico de barras: potência média por dia da semana
plt.figure(
    figsize=(10, 6)
)

plt.bar(
    daily_profile["Dia"],
    daily_profile["mean"]
)

plt.title(
    "Potência ativa média por dia da semana"
)

plt.xlabel(
    "Dia da semana"
)

plt.ylabel(
    "Potência ativa média (kW)"
)

plt.xticks(
    rotation=30
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/eda_dia_semana.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Tabela pivô: potência média por dia da semana x hora do dia
heatmap_data = (
    eda_pd
    .pivot_table(
        index="day_of_week",
        columns="hour_of_day",
        values="power",
        aggfunc="mean"
    )
    .sort_index()
)

heatmap_data.index = [
    day_names[i]
    for i in heatmap_data.index
]

display(heatmap_data)

In [0]:
# Heatmap: potência ativa média por hora e dia da semana
plt.figure(
    figsize=(14, 6)
)

plt.imshow(
    heatmap_data,
    aspect="auto"
)

plt.colorbar(
    label="Potência ativa média (kW)"
)

plt.xticks(
    ticks=range(24),
    labels=range(24)
)

plt.yticks(
    ticks=range(len(heatmap_data.index)),
    labels=heatmap_data.index
)

plt.title(
    "Potência ativa média por hora e dia da semana"
)

plt.xlabel(
    "Hora do dia"
)

plt.ylabel(
    "Dia da semana"
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/eda_heatmap_hora_dia.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
# Correlação de Pearson entre potência e lags (1h, 24h, 168h)
lag_correlations = pd.DataFrame({
    "Defasagem": [
        "1 hora",
        "24 horas",
        "168 horas"
    ],
    "Correlação": [
        eda_pd["power"].corr(
            eda_pd["lag_1"]
        ),
        eda_pd["power"].corr(
            eda_pd["lag_24"]
        ),
        eda_pd["power"].corr(
            eda_pd["lag_168"]
        )
    ]
})

display(lag_correlations)

In [0]:
# Gráfico de barras: correlação da potência com valores defasados
plt.figure(
    figsize=(8, 5)
)

plt.bar(
    lag_correlations["Defasagem"],
    lag_correlations["Correlação"]
)

plt.ylim(
    0,
    1
)

plt.title(
    "Correlação da potência ativa com valores defasados"
)

plt.xlabel(
    "Defasagem"
)

plt.ylabel(
    "Correlação de Pearson"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    f"{results_path}/eda_correlacao_lags.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

import sys
import pandas as pd
import numpy as np
import matplotlib
import sklearn
import statsmodels
import pyspark

arquivo = "/Volumes/workspace/default/tcc/household_power_consumption.txt"
pasta = "/Volumes/workspace/default/tcc/"

print("=" * 70)
print("1. METADADOS DO ARQUIVO")
print("=" * 70)

info = next(
    x for x in dbutils.fs.ls(pasta)
    if x.name == "household_power_consumption.txt"
)

print("Nome:", info.name)
print("Caminho:", info.path)
print("Tamanho em bytes:", info.size)
print("Tamanho em MB:", round(info.size / (1024 ** 2), 2))

if hasattr(info, "modificationTime"):
    print(
        "Data de modificação/upload:",
        datetime.fromtimestamp(
            info.modificationTime / 1000,
            tz=timezone.utc
        )
    )

print("\n" + "=" * 70)
print("2. PRIMEIRAS LINHAS DO ARQUIVO ORIGINAL")
print("=" * 70)

with open(arquivo, "r", encoding="utf-8") as f:
    for i in range(5):
        print(repr(f.readline().rstrip("\n")))

print("\n" + "=" * 70)
print("3. ESTRUTURA DO ARQUIVO")
print("=" * 70)

raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(arquivo)
)

print("Número de registros:", raw.count())
print("Número de colunas:", len(raw.columns))
print("Colunas:")
for i, c in enumerate(raw.columns, 1):
    print(f"{i}. {c}")

print("\nSchema lido:")
raw.printSchema()

print("\nPrimeiros registros:")
raw.show(5, truncate=False)

print("\n" + "=" * 70)
print("4. MARCADORES DE DADOS AUSENTES (?)")
print("=" * 70)

raw.select(
    *[
        F.sum(
            F.when(F.col(c) == "?", 1).otherwise(0)
        ).alias(c)
        for c in raw.columns
    ]
).show(truncate=False)

print("\n" + "=" * 70)
print("5. PERÍODO COBERTO PELOS DADOS")
print("=" * 70)

raw_ts = raw.withColumn(
    "timestamp",
    F.to_timestamp(
        F.concat_ws(" ", F.col("Date"), F.col("Time")),
        "d/M/yyyy HH:mm:ss"
    )
)

raw_ts.agg(
    F.min("timestamp").alias("data_inicial"),
    F.max("timestamp").alias("data_final")
).show(truncate=False)

print("\n" + "=" * 70)
print("6. AMBIENTE E VERSÕES")
print("=" * 70)

print("Python:", sys.version)
print("Spark:", spark.version)
print("PySpark:", pyspark.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("statsmodels:", statsmodels.__version__)

try:
    print(
        "Databricks Runtime:",
        spark.conf.get(
            "spark.databricks.clusterUsageTags.sparkVersion"
        )
    )
except:
    print("Databricks Runtime: não disponível via configuração")

In [0]:
# [OBSOLETO] Tentativa de verificar nulos sem reler o arquivo — falhou
# from pyspark.sql import functions as F

In [0]:
# ------------------------------------------------------------
# Verificação de valores ausentes no arquivo original
# Conta nulos, strings vazias e '?' por coluna
# ------------------------------------------------------------
from pyspark.sql import functions as F

# Lê novamente o arquivo original
raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(arquivo)
)

print("=" * 90)
print("VERIFICAÇÃO DOS VALORES AUSENTES NO ARQUIVO ORIGINAL")
print("=" * 90)

raw.select(
    *[
        F.struct(
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias("null"),
            
            F.sum(
                F.when(F.trim(F.col(c)) == "", 1).otherwise(0)
            ).alias("vazio"),
            
            F.sum(
                F.when(F.trim(F.col(c)) == "?", 1).otherwise(0)
            ).alias("interrogacao")
        ).alias(c)
        for c in raw.columns
    ]
).show(truncate=False)

In [0]:
print("=" * 90)
print("VALORES MAIS FREQUENTES EM Sub_metering_3")
print("=" * 90)

raw.groupBy("Sub_metering_3") \
   .count() \
   .orderBy(F.desc("count")) \
   .show(30, truncate=False)

print("=" * 90)
print("REGISTROS EM QUE Global_active_power ESTÁ AUSENTE")
print("=" * 90)

raw.filter(
    F.trim(F.col("Global_active_power")) == "?"
).select(
    "Date",
    "Time",
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
).show(20, truncate=False)